<a href="https://qworld.net" target="_blank" align="left"><img src="../../qworld/images/header.jpg"  align="left"></a>
$ \newcommand{\bra}[1]{\langle #1|} $
$ \newcommand{\ket}[1]{|#1\rangle} $
$ \newcommand{\braket}[2]{\langle #1|#2\rangle} $
$ \newcommand{\dot}[2]{ #1 \cdot #2} $
$ \newcommand{\biginner}[2]{\left\langle #1,#2\right\rangle} $
$ \newcommand{\mymatrix}[2]{\left( \begin{array}{#1} #2\end{array} \right)} $
$ \newcommand{\myvector}[1]{\mymatrix{c}{#1}} $
$ \newcommand{\myrvector}[1]{\mymatrix{r}{#1}} $
$ \newcommand{\mypar}[1]{\left( #1 \right)} $
$ \newcommand{\mybigpar}[1]{ \Big( #1 \Big)} $
$ \newcommand{\sqrttwo}{\frac{1}{\sqrt{2}}} $
$ \newcommand{\dsqrttwo}{\dfrac{1}{\sqrt{2}}} $
$ \newcommand{\onehalf}{\frac{1}{2}} $
$ \newcommand{\donehalf}{\dfrac{1}{2}} $
$ \newcommand{\hadamard}{ \mymatrix{rr}{ \sqrttwo & \sqrttwo \\ \sqrttwo & -\sqrttwo }} $
$ \newcommand{\vzero}{\myvector{1\\0}} $
$ \newcommand{\vone}{\myvector{0\\1}} $
$ \newcommand{\stateplus}{\myvector{ \sqrttwo \\  \sqrttwo } } $
$ \newcommand{\stateminus}{ \myrvector{ \sqrttwo \\ -\sqrttwo } } $
$ \newcommand{\myarray}[2]{ \begin{array}{#1}#2\end{array}} $
$ \newcommand{\X}{ \mymatrix{cc}{0 & 1 \\ 1 & 0}  } $
$ \newcommand{\I}{ \mymatrix{rr}{1 & 0 \\ 0 & 1}  } $
$ \newcommand{\Z}{ \mymatrix{rr}{1 & 0 \\ 0 & -1}  } $
$ \newcommand{\Htwo}{ \mymatrix{rrrr}{ \frac{1}{2} & \frac{1}{2} & \frac{1}{2} & \frac{1}{2} \\ \frac{1}{2} & -\frac{1}{2} & \frac{1}{2} & -\frac{1}{2} \\ \frac{1}{2} & \frac{1}{2} & -\frac{1}{2} & -\frac{1}{2} \\ \frac{1}{2} & -\frac{1}{2} & -\frac{1}{2} & \frac{1}{2} } } $
$ \newcommand{\CNOT}{ \mymatrix{cccc}{1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0} } $
$ \newcommand{\norm}[1]{ \left\lVert #1 \right\rVert } $
$ \newcommand{\pstate}[1]{ \lceil \mspace{-1mu} #1 \mspace{-1.5mu} \rfloor } $
$ \newcommand{\greenbit}[1] {\mathbf{{\color{green}#1}}} $
$ \newcommand{\bluebit}[1] {\mathbf{{\color{blue}#1}}} $
$ \newcommand{\redbit}[1] {\mathbf{{\color{red}#1}}} $
$ \newcommand{\brownbit}[1] {\mathbf{{\color{brown}#1}}} $
$ \newcommand{\blackbit}[1] {\mathbf{{\color{black}#1}}} $

# Solutions - Qiskit ML vs PennyLane

_Prepared by Claudia Zendejas-Morales_

This notebook contains complete solutions for the embedded tasks in notebook `1.2`.

In [1]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

import pennylane as qml

<a id="task1-solution"></a>

## Solution to Task 1

We reuse the same one-qubit construction in both frameworks, but now we evaluate it for several input values and two different values of the trainable parameter.

In [2]:
x_values = [0.0, 0.3, 0.6, 0.9]
theta_values = [0.0, 1.2]

def qiskit_p1(x, theta):
    qc = QuantumCircuit(1)
    qc.ry(2 * x, 0)
    qc.rz(theta, 0)
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()
    return float(probs.get("1", 0.0))

qiskit_results = {}
for theta in theta_values:
    qiskit_results[theta] = []
    for x in x_values:
        qiskit_results[theta].append(qiskit_p1(x, theta))

print("Qiskit results:")
for theta in theta_values:
    print(f"theta = {theta}: {qiskit_results[theta]}")

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def pl_circuit(x, theta):
    qml.RY(2 * x, wires=0)
    qml.RZ(theta, wires=0)
    return qml.probs(wires=0)

def pennylane_p1(x, theta):
    probs = pl_circuit(x, theta)
    return float(probs[1])

pennylane_results = {}
for theta in theta_values:
    pennylane_results[theta] = []
    for x in x_values:
        pennylane_results[theta].append(pennylane_p1(x, theta))

print("\nPennyLane results:")
for theta in theta_values:
    print(f"theta = {theta}: {pennylane_results[theta]}")

print("\nPairwise absolute differences:")
for theta in theta_values:
    for i, x in enumerate(x_values):
        diff = abs(qiskit_results[theta][i] - pennylane_results[theta][i])
        print(f"x = {x}, theta = {theta}: {diff}")

print("\nEffect of changing theta inside each framework:")
for i, x in enumerate(x_values):
    qiskit_theta_diff = abs(qiskit_results[theta_values[0]][i] - qiskit_results[theta_values[1]][i])
    pennylane_theta_diff = abs(pennylane_results[theta_values[0]][i] - pennylane_results[theta_values[1]][i])
    print(f"x = {x}: Qiskit diff = {qiskit_theta_diff}, PennyLane diff = {pennylane_theta_diff}")

Qiskit results:
theta = 0.0: [0.0, 0.08733219254516084, 0.31882112276166324, 0.6136010473465435]
theta = 1.2: [0.0, 0.08733219254516084, 0.31882112276166324, 0.6136010473465437]

PennyLane results:
theta = 0.0: [0.0, 0.08733219254516084, 0.31882112276166324, 0.6136010473465435]
theta = 1.2: [0.0, 0.08733219254516084, 0.3188211227616633, 0.6136010473465436]

Pairwise absolute differences:
x = 0.0, theta = 0.0: 0.0
x = 0.3, theta = 0.0: 0.0
x = 0.6, theta = 0.0: 0.0
x = 0.9, theta = 0.0: 0.0
x = 0.0, theta = 1.2: 0.0
x = 0.3, theta = 1.2: 0.0
x = 0.6, theta = 1.2: 5.551115123125783e-17
x = 0.9, theta = 1.2: 1.1102230246251565e-16

Effect of changing theta inside each framework:
x = 0.0: Qiskit diff = 0.0, PennyLane diff = 0.0
x = 0.3: Qiskit diff = 0.0, PennyLane diff = 0.0
x = 0.6: Qiskit diff = 0.0, PennyLane diff = 5.551115123125783e-17
x = 0.9: Qiskit diff = 2.220446049250313e-16, PennyLane diff = 1.1102230246251565e-16


For each pair `(x, theta)`, the Qiskit and PennyLane outputs should match up to floating-point precision because both frameworks implement the same one-qubit unitary transformation followed by the same measurement. The interesting observation is that in this specific example changing `theta` does not change $P(1)$, while changing `x` does. This happens because $R_Y(2x)$ changes the amplitudes, but $R_Z(\theta)$ only changes a relative phase that is not visible when we measure probabilities in the computational basis.

<a id="task2-solution"></a>

## Solution to Task 2

We keep the same circuit structure, but we change the output from $P(1)$ to the expectation value $\langle Z \rangle$.

In [3]:
x = 0.6
theta = 1.2

# Qiskit solution
qc = QuantumCircuit(1)
qc.ry(2 * x, 0)
qc.rz(theta, 0)

sv = Statevector.from_instruction(qc)
qiskit_probs = sv.probabilities_dict()
qiskit_p1 = float(qiskit_probs.get("1", 0.0))
qiskit_z = float(qiskit_probs.get("0", 0.0) - qiskit_probs.get("1", 0.0))

print("Qiskit probabilities:", qiskit_probs)
print("Qiskit <Z>:", qiskit_z)
print("Qiskit check: 1 - 2P(1):", 1 - 2 * qiskit_p1)

# PennyLane solution
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def pl_probs_circuit(x, theta):
    qml.RY(2 * x, wires=0)
    qml.RZ(theta, wires=0)
    return qml.probs(wires=0)

@qml.qnode(dev)
def pl_z_circuit(x, theta):
    qml.RY(2 * x, wires=0)
    qml.RZ(theta, wires=0)
    return qml.expval(qml.Z(0))

pl_probs = pl_probs_circuit(x, theta)
pl_p1 = float(pl_probs[1])
pl_z = float(pl_z_circuit(x, theta))

print("\nPennyLane probabilities:", pl_probs)
print("PennyLane <Z>:", pl_z)
print("PennyLane check: 1 - 2P(1):", 1 - 2 * pl_p1)

print("\nAbsolute difference in <Z>:", abs(qiskit_z - pl_z))

Qiskit probabilities: {np.str_('0'): np.float64(0.6811788772383368), np.str_('1'): np.float64(0.31882112276166324)}
Qiskit <Z>: 0.36235775447667357
Qiskit check: 1 - 2P(1): 0.3623577544766735

PennyLane probabilities: [0.68117888 0.31882112]
PennyLane <Z>: 0.3623577544766736
PennyLane check: 1 - 2P(1): 0.3623577544766734

Absolute difference in <Z>: 5.551115123125783e-17


The key observation is that the circuit itself does not need to change; only the measured quantity changes. In Qiskit, we chose to compute $\langle Z \rangle$ from the probabilities obtained from the statevector. In PennyLane, we changed the `QNode` measurement from `qml.probs` to `qml.expval(qml.Z(0))`, which makes the measurement choice more explicit at the function level.

<a id="task3-solution"></a>

## Solution to Task 3

One reasonable set of answers is:

1. **Quantum-kernel or QSVM workflow with Qiskit circuits**: `Qiskit Machine Learning`.
   - It is directly aligned with Qiskit circuit objects and Qiskit-native kernel abstractions.
   - It fits naturally when the rest of the workflow already depends on Qiskit simulators or IBM Quantum backends.

2. **Differentiable hybrid model to be connected to a deep-learning workflow**: `PennyLane`.
   - PennyLane is designed around differentiable quantum programming, so measurement outputs integrate naturally with optimization pipelines.
   - It is often more convenient when the main engineering need is auto-diff-based training rather than Qiskit-native kernel tooling.

3. **Existing Qiskit project plus a new variational classifier or kernel experiment**: `Qiskit Machine Learning`.
   - Reusing the same circuit objects, backend flow, and surrounding Qiskit ecosystem usually reduces engineering overhead.
   - Qiskit Machine Learning already exposes model families such as kernels and variational classifiers in a way that matches that ecosystem.

4. **Notebook prototype where the same circuit may return probabilities now and expectation values later**: `PennyLane`.
   - In PennyLane, the measurement choice is explicit in the `QNode`, so changing from `qml.probs` to `qml.expval(...)` is a very natural workflow.
   - This style is especially convenient when iterating quickly on hybrid models and trying several output definitions.

5. **Short teaching or exploratory notebook with no strong ecosystem constraint yet**: `both acceptable`.
   - At small scale, either framework can express the main QML idea clearly enough for learning purposes.
   - In that situation, readability, available examples, and the instructor's preferred workflow may matter more than framework-specific features.

An alternative is possible in some of these cases, but the justification should still mention concrete workflow tradeoffs rather than generic preferences.